This snippet covers data loading, column normalization, and missing value imputation:

* **Library Imports:** Loads essential packages for numerical computation (`numpy`), tabular data handling (`pandas`), visualization (`matplotlib`, `seaborn`), Hugging Face dataset loading (`datasets`), machine learning model evaluation, scaling, encoding, and ensemble classifiers (`sklearn`, `xgboost`, `catboost`, `lightgbm`).
* **Dataset Loading & Header Cleanup:** Loads the customer segmentation dataset from Hugging Face, converts the training split into a Pandas DataFrame, and strips surrounding whitespace from column names.
* **Missing Data Imputation:** Imputes missing categorical values (`Ever_Married`, `Graduated`, `Profession`) with the label `'Unknown'` and fills missing numerical values in `Work_Experience` using its median.

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, precision_recall_fscore_support

from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

# ==========================================
# 1. DATA LOADING & CLEANING
# ==========================================

# Load dataset from Hugging Face and convert to pandas DataFrame
ds = load_dataset("jason1966/abisheksudarshan_customer-segmentation")
df = ds['train'].to_pandas()

# Strip extra spaces from column names
df.columns = [c.strip() for c in df.columns]

# Impute missing values for categorical and numerical features
df['Ever_Married'] = df['Ever_Married'].fillna('Unknown')
df['Graduated'] = df['Graduated'].fillna('Unknown')
df['Profession'] = df['Profession'].fillna('Unknown')
df['Work_Experience'] = df['Work_Experience'].fillna(df['Work_Experience'].median())


This snippet performs feature engineering by creating interaction terms, numerical ratios, binned categorical variables, and conditional features:

* **DataFrame Copying:** Creates a independent copy (`df_fe`) of the cleaned DataFrame to preserve the original structure.
* **Ratio & Numerical Interactions:** Calculates `Experience_Age_Ratio` to represent career duration relative to age, converts `Ever_Married` into a binary integer (`Is_Married`), and creates an interaction feature (`Age_Married_Interact`).
* **Continuous Feature Binning:** Uses `pd.cut()` to discretize continuous values into ordinal categorical groups, converting `Age` into four life stages (`Age_Group`) and `Work_Experience` into four experience levels (`Career_Stage`).
* **Education & Combined Categoricals:** Converts graduation status to binary format (`Graduated_Int`), calculates its interaction with age (`Age_Graduated_Interact`), and combines marital status with profession into a joint categorical column (`Married_Profession`).
* **Conditional Feature Engineering:** Checks for the presence of `Family_Size` in the dataset and, if present, creates `Age_Per_Family_Member` to capture household age density.

In [10]:

# ==========================================
# 2. FEATURE ENGINEERING
# ==========================================

df_fe = df.copy()

# Ratio and interaction features
df_fe['Experience_Age_Ratio'] = df_fe['Work_Experience'] / df_fe['Age']
df_fe['Is_Married'] = (df_fe['Ever_Married'] == 'Yes').astype(int)
df_fe['Age_Married_Interact'] = df_fe['Age'] * df_fe['Is_Married']

# Binning continuous variables into categorical groups
df_fe['Age_Group'] = pd.cut(df_fe['Age'], bins=[0, 25, 40, 60, 100], labels=['Young', 'Adult', 'Mid_Age', 'Senior'])
df_fe['Career_Stage'] = pd.cut(df_fe['Work_Experience'], bins=[-1, 1, 5, 10, 50], labels=['Junior', 'Mid', 'Senior', 'Veteran'])

# Education and interaction features
df_fe['Graduated_Int'] = (df_fe['Graduated'] == 'Yes').astype(int)
df_fe['Age_Graduated_Interact'] = df_fe['Age'] * df_fe['Graduated_Int']
df_fe['Married_Profession'] = df_fe['Ever_Married'].astype(str) + "_" + df_fe['Profession'].astype(str)

# Family size derived feature (if present in dataset)
if 'Family_Size' in df_fe.columns:
    df_fe['Age_Per_Family_Member'] = df_fe['Age'] / (df_fe['Family_Size'] + 1)


This snippet handles feature selection, label encoding, dummy encoding, stratified data splitting, and numerical feature scaling:

* **Feature & Target Separation:** Identifies and drops target and identifier columns (`Spending_Score`, `Customer_ID`, `ID`) from `df_fe` to construct the feature matrix `X`, storing raw target values in `y_raw`.
* **Target Label Encoding:** Uses `LabelEncoder()` to map categorical classes of the target variable `Spending_Score` to integer indices (`y`) and preserves original class names in `target_names`.
* **Categorical One-Hot Encoding:** Identifies all text and category dtype columns using `select_dtypes()` and converts them into binary dummy variables via `pd.get_dummies()`, applying `drop_first=True` to eliminate redundant features.
* **Stratified Data Splitting:** Partitions the encoded dataset into an 80% training set and a 20% test set using `train_test_split()`, with `stratify=y` ensuring balanced class proportions across splits.
* **Feature Standardization:** Uses `StandardScaler()` to scale numerical columns (`num_cols`) to zero mean and unit variance, fitting exclusively on `X_train` to prevent data leakage and applying the transformation to both `X_train` and `X_test`.

<!-- --- Random Forest Classification Report ---
              precision    recall  f1-score   support

     Average       0.57      0.90      0.70       395
        High       0.66      0.55      0.60       243
         Low       0.93      0.75      0.83       976

    accuracy                           0.76      1614
   macro avg       0.72      0.73      0.71      1614
weighted avg       0.80      0.76      0.76      1614 -->

In [11]:

# ==========================================
# 3. PREPROCESSING & ENCODING
# ==========================================

# Separate features (X) and target variable (y)
features_to_drop = ['Spending_Score', 'Customer_ID', 'ID']
features_to_drop = [col for col in features_to_drop if col in df_fe.columns]

X = df_fe.drop(columns=features_to_drop)
y_raw = df_fe['Spending_Score']

# Encode target categorical labels into numerical values
target_encoder = LabelEncoder()
y = target_encoder.fit_transform(y_raw)
target_names = target_encoder.classes_

# One-hot encode categorical feature columns
cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)

# Split data into train and test sets (80/20) with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

# Standardize numerical features
num_cols = X_encoded.select_dtypes(include=['int64', 'float64', 'int32']).columns.tolist()
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])


This snippet trains and evaluates a Random Forest classifier model:

* **Model Training:** Instantiates a `RandomForestClassifier` configured with 150 decision trees, a maximum tree depth of 10, and parallel processing (`n_jobs=-1`), then fits it on the training data (`X_train`, `y_train`).
* **Prediction Generation:** Uses the trained Random Forest model (`rf_model`) to generate predictions (`y_pred_rf`) on the unseen test dataset (`X_test`).
* **Metric Computation:** Calculates overall classification accuracy using `accuracy_score()` alongside weighted precision, recall, and F1-score using `precision_recall_fscore_support()`.
* **Results Output:** Prints formatted summary statistics and a detailed class-level evaluation report using `classification_report()` mapped to original target class names.

In [12]:

# ==========================================
# 4. RANDOM FOREST MODEL
# ==========================================

rf_model = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

# Evaluate Random Forest
rf_acc = accuracy_score(y_test, y_pred_rf)
rf_p, rf_r, rf_f1, _ = precision_recall_fscore_support(y_test, y_pred_rf, average='weighted')

print("\n" + "="*45)
print("          RANDOM FOREST RESULTS          ")
print("="*45)
print(f" Accuracy : {rf_acc:.4f}")
print(f" Precision: {rf_p:.4f}")
print(f" Recall    : {rf_r:.4f}")
print(f" F1-Score  : {rf_f1:.4f}")
print("\n--- Random Forest Classification Report ---")
print(classification_report(y_test, y_pred_rf, target_names=target_names))


          RANDOM FOREST RESULTS          
 Accuracy : 0.8197
 Precision: 0.8437
 Recall    : 0.8197
 F1-Score  : 0.8234

--- Random Forest Classification Report ---
              precision    recall  f1-score   support

     Average       0.64      0.89      0.74       395
        High       0.74      0.58      0.65       243
         Low       0.95      0.85      0.90       976

    accuracy                           0.82      1614
   macro avg       0.78      0.77      0.76      1614
weighted avg       0.84      0.82      0.82      1614



This snippet trains and evaluates an XGBoost gradient boosting classifier:

* **Model Initialization & Training:** Instantiates an `XGBClassifier` configured with 150 boosting rounds, a learning rate of 0.05, a maximum depth of 5, feature/row subsampling rates of 0.8, and multiclass log loss (`mlogloss`) evaluation metric, then fits it on `X_train` and `y_train`.
* **Test Prediction:** Generates multi-class target predictions (`y_pred_xgb`) for the test set (`X_test`).
* **Performance Evaluation:** Computes test accuracy, weighted precision, weighted recall, and weighted F1-score using `accuracy_score()` and `precision_recall_fscore_support()`.
* **Results Output:** Prints formatted metric summaries alongside a comprehensive class-level classification report using original target class names (`target_names`).

In [13]:

# ==========================================
# 5. XGBOOST MODEL
# ==========================================

xgb_model = XGBClassifier(
    n_estimators=150, 
    learning_rate=0.05, 
    max_depth=5, 
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42, 
    eval_metric='mlogloss'
)
xgb_model.fit(X_train, y_train)
y_pred_xgb = xgb_model.predict(X_test)

# Evaluate XGBoost
xgb_acc = accuracy_score(y_test, y_pred_xgb)
xgb_p, xgb_r, xgb_f1, _ = precision_recall_fscore_support(y_test, y_pred_xgb, average='weighted')

print("\n" + "="*45)
print("              XGBOOST RESULTS             ")
print("="*45)
print(f" Accuracy : {xgb_acc:.4f}")
print(f" Precision: {xgb_p:.4f}")
print(f" Recall    : {xgb_r:.4f}")
print(f" F1-Score  : {xgb_f1:.4f}")
print("\n--- XGBoost Classification Report ---")
print(classification_report(y_test, y_pred_xgb, target_names=target_names))



              XGBOOST RESULTS             
 Accuracy : 0.8166
 Precision: 0.8286
 Recall    : 0.8166
 F1-Score  : 0.8179

--- XGBoost Classification Report ---
              precision    recall  f1-score   support

     Average       0.66      0.85      0.74       395
        High       0.73      0.58      0.64       243
         Low       0.92      0.86      0.89       976

    accuracy                           0.82      1614
   macro avg       0.77      0.76      0.76      1614
weighted avg       0.83      0.82      0.82      1614



This snippet trains and evaluates a LightGBM gradient boosting classifier model:

* **Model Training:** Instantiates an `LGBMClassifier` configured with 150 estimators, a learning rate of 0.05, a maximum depth of 5, and suppressed console output (`verbose=-1`), then fits it on `X_train` and `y_train`.
* **Prediction Generation:** Uses the trained LightGBM model (`lgb_model`) to predict class labels (`y_pred_lgb`) on the test feature set (`X_test`).
* **Metric Computation:** Calculates overall classification accuracy via `accuracy_score()` along with weighted precision, recall, and F1-score via `precision_recall_fscore_support()`.
* **Results Output:** Prints formatted summary statistics and a detailed class-level evaluation report using `classification_report()` mapped to original target class names (`target_names`).

In [14]:
# ==========================================
# 6. LIGHTGBM MODEL
# ==========================================

lgb_model = LGBMClassifier(
    n_estimators=150, 
    learning_rate=0.05, 
    max_depth=5, 
    random_state=42, 
    verbose=-1
)
lgb_model.fit(X_train, y_train)
y_pred_lgb = lgb_model.predict(X_test)

# Evaluate LightGBM
lgb_acc = accuracy_score(y_test, y_pred_lgb)
lgb_p, lgb_r, lgb_f1, _ = precision_recall_fscore_support(y_test, y_pred_lgb, average='weighted')

print("\n" + "="*45)
print("              LIGHTGBM RESULTS            ")
print("="*45)
print(f" Accuracy : {lgb_acc:.4f}")
print(f" Precision: {lgb_p:.4f}")
print(f" Recall    : {lgb_r:.4f}")
print(f" F1-Score  : {lgb_f1:.4f}")
print("\n--- LightGBM Classification Report ---")
print(classification_report(y_test, y_pred_lgb, target_names=target_names))



              LIGHTGBM RESULTS            
 Accuracy : 0.8178
 Precision: 0.8287
 Recall    : 0.8178
 F1-Score  : 0.8187

--- LightGBM Classification Report ---
              precision    recall  f1-score   support

     Average       0.67      0.85      0.74       395
        High       0.73      0.58      0.65       243
         Low       0.92      0.87      0.89       976

    accuracy                           0.82      1614
   macro avg       0.77      0.76      0.76      1614
weighted avg       0.83      0.82      0.82      1614



This snippet trains and evaluates a CatBoost gradient boosting classifier model:

* **Model Training:** Instantiates a `CatBoostClassifier` configured with 150 iterations, a learning rate of 0.05, a tree depth of 5, and silent fitting execution (`verbose=0`), then fits it on `X_train` and `y_train`.
* **Prediction Generation:** Uses the trained CatBoost model (`cb_model`) to predict class labels (`y_pred_cb`) on the test feature set (`X_test`).
* **Metric Computation:** Calculates overall classification accuracy via `accuracy_score()` alongside weighted precision, recall, and F1-score via `precision_recall_fscore_support()`.
* **Results Output:** Prints formatted summary statistics and a detailed class-level evaluation report using `classification_report()` mapped to original target class names (`target_names`).

In [15]:

# ==========================================
# 7. CATBOOST MODEL
# ==========================================

cb_model = CatBoostClassifier(
    iterations=150, 
    learning_rate=0.05, 
    depth=5, 
    random_seed=42, 
    verbose=0
)
cb_model.fit(X_train, y_train)
y_pred_cb = cb_model.predict(X_test)

# Evaluate CatBoost
cb_acc = accuracy_score(y_test, y_pred_cb)
cb_p, cb_r, cb_f1, _ = precision_recall_fscore_support(y_test, y_pred_cb, average='weighted')

print("\n" + "="*45)
print("              CATBOOST RESULTS            ")
print("="*45)
print(f" Accuracy : {cb_acc:.4f}")
print(f" Precision: {cb_p:.4f}")
print(f" Recall    : {cb_r:.4f}")
print(f" F1-Score  : {cb_f1:.4f}")
print("\n--- CatBoost Classification Report ---")
print(classification_report(y_test, y_pred_cb, target_names=target_names))





              CATBOOST RESULTS            
 Accuracy : 0.8191
 Precision: 0.8366
 Recall    : 0.8191
 F1-Score  : 0.8219

--- CatBoost Classification Report ---
              precision    recall  f1-score   support

     Average       0.65      0.86      0.74       395
        High       0.75      0.60      0.67       243
         Low       0.93      0.86      0.89       976

    accuracy                           0.82      1614
   macro avg       0.78      0.77      0.77      1614
weighted avg       0.84      0.82      0.82      1614



This snippet automates model training, evaluation, and comparative benchmarking across individual classifiers and the Hard Voting Ensemble:

* **Model & Ensemble Initialization:** Configures four base estimators (`RandomForest`, `ExtraTrees`, `XGBoost`, `LightGBM`) and encapsulates them inside a majority-rule `VotingClassifier(voting='hard')`, storing all candidate models in a dictionary for streamlined iteration.
* **Automated Benchmarking Loop:** Fits each model on `X_train`, generates flattened predictions on `X_test`, and calculates key classification metrics (`Accuracy`, weighted `Precision`, `Recall`, and `F1-Score`).
* **Leaderboard Construction:** Aggregates metrics into a pandas DataFrame, sorts performance descending by F1-Score and Accuracy, and prints a formatted percentage-based leaderboard table.
* **Best Model Identification:** Programmatically extracts the top-performing model and displays its winning accuracy and F1-score metrics, followed by a detailed class-level `classification_report` for the Hard Voting Ensemble.

In [17]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

# ==========================================
# 1. INITIALIZE BASE MODELS & HARD VOTING ENSEMBLE
# ==========================================
rf_model = RandomForestClassifier(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1)
et_model = ExtraTreesClassifier(n_estimators=150, max_depth=10, random_state=42, n_jobs=-1)
xgb_model = XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=42, eval_metric='mlogloss')
lgb_model = LGBMClassifier(n_estimators=150, learning_rate=0.05, max_depth=5, random_state=42, verbose=-1)

# Build Hard Voting Ensemble
voting_hard_model = VotingClassifier(
    estimators=[
        ('rf', rf_model),
        ('et', et_model),
        ('xgb', xgb_model),
        ('lgbm', lgb_model)
    ],
    voting='hard'
)

# Model dictionary including base models and the ensemble
all_models = {
    "Random Forest": rf_model,
    "Extra Trees": et_model,
    "XGBoost": xgb_model,
    "LightGBM": lgb_model,
    "Hard Voting Ensemble": voting_hard_model
}

# ==========================================
# 2. FIT & EVALUATE ALL MODELS
# ==========================================
comparison_list = []

for name, model in all_models.items():
    # Fit model on training set
    model.fit(X_train, y_train)
    
    # Predict on test set
    preds = model.predict(X_test)
    if hasattr(preds, 'ndim') and preds.ndim > 1:
        preds = preds.ravel()
        
    # Calculate performance metrics
    acc = accuracy_score(y_test, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(y_test, preds, average='weighted')
    
    comparison_list.append({
        'Model': name,
        'Accuracy': acc,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    })

# ==========================================
# 3. DISPLAY LEADERBOARD & WINNING MODEL
# ==========================================
comparison_df = pd.DataFrame(comparison_list).sort_values(by=['F1-Score', 'Accuracy'], ascending=False).reset_index(drop=True)

print("="*65)
print("       MODEL & HARD VOTING ENSEMBLE COMPARISON LEADERBOARD")
print("="*65)
print(comparison_df.to_string(index=False, float_format=lambda x: f"{x*100:.2f}%"))

# Identify top model
best_model_name = comparison_df.iloc[0]['Model']
best_acc = comparison_df.iloc[0]['Accuracy'] * 100
best_f1 = comparison_df.iloc[0]['F1-Score'] * 100

print("\n" + "="*65)
print(f"🏆 WINNING MODEL: {best_model_name}")
print(f"   • Top Accuracy : {best_acc:.2f}%")
print(f"   • Top F1-Score : {best_f1:.2f}%")
print("="*65)

# Classification report for the Hard Voting model specifically
y_pred_hard = voting_hard_model.predict(X_test)
if hasattr(y_pred_hard, 'ndim') and y_pred_hard.ndim > 1:
    y_pred_hard = y_pred_hard.ravel()

target_names = target_encoder.classes_
print(f"\n--- Detailed Classification Report (Hard Voting Ensemble) ---")
print(classification_report(y_test, y_pred_hard, target_names=target_names))

       MODEL & HARD VOTING ENSEMBLE COMPARISON LEADERBOARD
               Model  Accuracy  Precision  Recall  F1-Score
Hard Voting Ensemble    82.28%     84.53%  82.28%    82.60%
       Random Forest    81.97%     84.37%  81.97%    82.34%
             XGBoost    81.85%     83.17%  81.85%    82.03%
            LightGBM    81.78%     82.87%  81.78%    81.87%
         Extra Trees    81.47%     83.41%  81.47%    81.72%

🏆 WINNING MODEL: Hard Voting Ensemble
   • Top Accuracy : 82.28%
   • Top F1-Score : 82.60%

--- Detailed Classification Report (Hard Voting Ensemble) ---
              precision    recall  f1-score   support

     Average       0.65      0.90      0.75       395
        High       0.74      0.59      0.66       243
         Low       0.95      0.85      0.90       976

    accuracy                           0.82      1614
   macro avg       0.78      0.78      0.77      1614
weighted avg       0.85      0.82      0.83      1614

